# 07 테스트 리뷰별 이벤트 예측 + 별점 정제 비교

07번에서는 test 리뷰를 각 모델에 넣었을 때 이벤트성 리뷰인지 먼저 확인하고, 그 예측이 후보 1/2 별점 정제에 어떻게 반영되는지까지 연결해서 확인한다.

전체 모델 4개에 대해 계산은 모두 수행하되, 최종 해석은 아래 두 모델 중심으로 진행한다.

- 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)
- 선택 모델 (Metadata only + MLP)

정제 방식은 다음 두 가지다.

1. 후보 1: 이벤트 리뷰로 예측된 리뷰 제외 후 가게별 별점 평균
2. 후보 2: 이벤트 확률 기반 가중 평균

즉, 전체 별점 정제 계산은 `4개 모델 x 2개 후보 = 8개 결과`로 수행한다.


## 1. 라이브러리 로드

저장된 모델을 불러오고, test split 기준으로 리뷰별 예측과 가게별 별점 정제를 비교한다.


In [45]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)


## 2. test 데이터와 원본 리뷰 데이터 연결

05/06번과 같은 방식으로 test split을 재현한다.

`final_hybrid_test.csv`와 원본 리뷰의 label 순서가 일치하는지 검증한 뒤, test 리뷰에 원본 별점과 가게 정보를 붙여 리뷰별 예측 및 별점 정제에 사용한다. `store_name`이 비어 있는 행은 `store_url`을 가게 식별자로 사용한다.


In [46]:
SEED = 42
TARGET_COL = 'target_label'
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
REVIEW_TEXT_COL = 'review_text'
STORE_COL = 'store_name'
STORE_URL_COL = 'store_url'
STORE_ID_COL = 'store_id'
RATING_COL = 'rating'

train_df = pd.read_csv('csv/final_hybrid_train.csv')
val_df = pd.read_csv('csv/final_hybrid_val.csv')
test_df = pd.read_csv('csv/final_hybrid_test.csv')
raw_df = pd.read_csv('csv/preprocessed_reviews.csv')

y_test = test_df[TARGET_COL].astype(int)

train_idx, temp_idx = train_test_split(
    raw_df.index,
    test_size=0.3,
    random_state=SEED,
    stratify=raw_df[LABEL_COL],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=raw_df.loc[temp_idx, LABEL_COL],
)

np.testing.assert_array_equal(raw_df.loc[test_idx, LABEL_COL].astype(int).to_numpy(), y_test.to_numpy())

test_review_df = raw_df.loc[test_idx].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df[RATING_COL] = pd.to_numeric(test_review_df[RATING_COL], errors='coerce')
test_review_df[STORE_ID_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_ID_COL] = test_review_df[STORE_ID_COL].where(
    test_review_df[STORE_ID_COL].ne(''),
    test_review_df[STORE_URL_COL].fillna('').astype(str),
)
test_review_df[STORE_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_COL] = test_review_df[STORE_COL].where(
    test_review_df[STORE_COL].ne(''),
    test_review_df[STORE_ID_COL],
)

text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('test feature shape:', test_df.shape)
print('test review shape:', test_review_df.shape)
print('test label 분포:', y_test.value_counts().sort_index().to_dict())
print('test 가게 수:', test_review_df[STORE_ID_COL].nunique())


test feature shape: (1326, 148)
test review shape: (1326, 17)
test label 분포: {0: 853, 1: 473}
test 가게 수: 118


## 3. 전체 모델 4개 로드

07번에서 비교할 모델은 다음 네 가지다.

1. TF-IDF + Random Forest
2. KcBERT PCA only + MLP
3. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)
4. 선택 모델 (Metadata only + MLP)


In [47]:
model_specs = [
    {
        'model_key': 'baseline_tfidf_random_forest',
        'model_label': 'TF-IDF + Random Forest',
        'model_role': 'baseline',
        'path': 'outputs/baseline_tfidf_random_forest_model.joblib',
        'input_type': 'text',
    },
    {
        'model_key': 'ablation_mlp_kcbert_pca_only',
        'model_label': 'KcBERT PCA only + MLP',
        'model_role': 'ablation',
        'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model_key': 'proposed_hybrid_mlp_04',
        'model_label': '제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)',
        'model_role': 'proposed',
        'path': 'outputs/proposed_mlp_final_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model_key': 'final_selected_metadata_mlp',
        'model_label': '선택 모델 (Metadata only + MLP)',
        'model_role': 'selected',
        'path': 'outputs/final_selected_model.joblib',
        'input_type': 'tabular',
    },
]

missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'필요한 모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model_key']: joblib.load(spec['path']) for spec in model_specs}

loaded_models = pd.DataFrame([
    {
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'path': spec['path'],
    }
    for spec in model_specs
])
display(loaded_models)


,model_key,model_label,model_role,path
0,baseline_tfidf_random_forest,TF-IDF + Random Forest,baseline,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,KcBERT PCA only + MLP,ablation,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,proposed_hybrid_mlp_04,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),proposed,outputs/proposed_mlp_final_model.joblib
3,final_selected_metadata_mlp,선택 모델 (Metadata only + MLP),selected,outputs/final_selected_model.joblib


## 4. test 리뷰별 이벤트 확률 및 예측 계산

각 모델에서 test 리뷰별 이벤트 확률, threshold 기준 이벤트 예측 여부, 실제 라벨과의 정답 여부를 계산한다.

여기서부터 리뷰 단위로 `이 리뷰가 이벤트성 리뷰인지 아닌지`, `실제로 맞췄는지`, `후보 1,2에 어떻게 반영되는지`를 확인할 수 있다.


In [48]:
def predict_event_probability(spec, bundle):
    model = bundle['model']
    if spec['input_type'] == 'text':
        return model.predict_proba(text_test)[:, 1]

    feature_cols = bundle.get('feature_cols')
    if feature_cols is None:
        raise ValueError(f"{spec['model_label']} bundle에 feature_cols가 없습니다.")
    return model.predict_proba(test_df[feature_cols])[:, 1]


def make_error_type(actual, pred):
    conditions = [
        (actual == 1) & (pred == 1),
        (actual == 0) & (pred == 0),
        (actual == 0) & (pred == 1),
        (actual == 1) & (pred == 0),
    ]
    return np.select(conditions, ['TP', 'TN', 'FP', 'FN'], default='UNKNOWN')


prediction_frames = []
for spec in model_specs:
    bundle = model_bundles[spec['model_key']]
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    event_prob = predict_event_probability(spec, bundle)
    event_pred = (event_prob >= threshold).astype(int)
    actual_label = y_test.to_numpy()
    is_correct = actual_label == event_pred
    candidate_2_weight = np.clip(1 - event_prob, 0, 1)

    prediction_frames.append(pd.DataFrame({
        'raw_index': test_review_df['raw_index'].to_numpy(),
        'store_id': test_review_df[STORE_ID_COL].to_numpy(),
        'store_name': test_review_df[STORE_COL].to_numpy(),
        'store_url': test_review_df[STORE_URL_COL].to_numpy(),
        'rating': test_review_df[RATING_COL].to_numpy(),
        'review_text': test_review_df[REVIEW_TEXT_COL].fillna('').astype(str).to_numpy(),
        'cleaned_review_text': test_review_df[TEXT_COL].fillna('').astype(str).to_numpy(),
        'actual_label': actual_label,
        'actual_label_name': np.where(actual_label == 1, '이벤트 리뷰(1)', '일반 리뷰(0)'),
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'threshold': threshold,
        'event_prob': event_prob,
        'event_pred': event_pred,
        'event_pred_name': np.where(event_pred == 1, '이벤트 예측(1)', '일반 예측(0)'),
        'is_correct': is_correct,
        'error_type': make_error_type(actual_label, event_pred),
        'candidate_1_action': np.where(event_pred == 1, '제외', '포함'),
        'candidate_2_weight': candidate_2_weight,
        'candidate_2_weighted_rating': test_review_df[RATING_COL].to_numpy() * candidate_2_weight,
    }))

prediction_detail = pd.concat(prediction_frames, ignore_index=True)

expected_rows = len(test_review_df) * len(model_specs)
assert len(prediction_detail) == expected_rows, f'리뷰별 예측 행 수가 맞지 않습니다: {len(prediction_detail)} != {expected_rows}'
assert prediction_detail['is_correct'].equals(prediction_detail['actual_label'].eq(prediction_detail['event_pred']))
np.testing.assert_allclose(
    prediction_detail['candidate_2_weight'].to_numpy(),
    1 - prediction_detail['event_prob'].to_numpy(),
)
assert prediction_detail.loc[prediction_detail['event_pred'] == 1, 'candidate_1_action'].eq('제외').all()
assert prediction_detail.loc[prediction_detail['event_pred'] == 0, 'candidate_1_action'].eq('포함').all()

prediction_summary = (
    prediction_detail.groupby(['model_key', 'model_label', 'model_role', 'threshold'])
    .agg(
        review_count=('event_pred', 'size'),
        event_pred_count=('event_pred', 'sum'),
        event_pred_rate=('event_pred', 'mean'),
        correct_count=('is_correct', 'sum'),
        accuracy=('is_correct', 'mean'),
        mean_event_prob=('event_prob', 'mean'),
    )
    .reset_index()
    .round({
        'threshold': 4,
        'event_pred_rate': 4,
        'accuracy': 4,
        'mean_event_prob': 4,
    })
)

display(prediction_summary)


,model_key,model_label,model_role,threshold,review_count,event_pred_count,event_pred_rate,correct_count,accuracy,mean_event_prob
0,ablation_mlp_kcbert_pca_only,KcBERT PCA only + MLP,ablation,0.5038,1326,525,0.3959,718,0.5415,0.4227
1,baseline_tfidf_random_forest,TF-IDF + Random Forest,baseline,0.5021,1326,200,0.1508,821,0.6192,0.3534
2,final_selected_metadata_mlp,선택 모델 (Metadata only + MLP),selected,0.5004,1326,843,0.6357,702,0.5294,0.4965
3,proposed_hybrid_mlp_04,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),proposed,0.5074,1326,580,0.4374,723,0.5452,0.4413


## 5. 리뷰 10개 이벤트 판별 예시

test 리뷰 중 10개를 예시로 뽑아, 제안 모델과 선택 모델이 각각 이벤트성 리뷰로 판단했는지 확인한다.

전체 상세 컬럼은 `main_review_predictions_detail` 변수에 남기고, 화면에는 예시 리뷰만 출력한다.


In [49]:
review_detail_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'threshold',
    'is_correct',
    'error_type',
    'candidate_1_action',
    'candidate_2_weight',
    'candidate_2_weighted_rating',
]

main_review_predictions = (
    prediction_detail[prediction_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_predictions_detail = main_review_predictions[review_detail_cols].round({
    'event_prob': 4,
    'threshold': 4,
    'candidate_2_weight': 4,
    'candidate_2_weighted_rating': 4,
})

EXAMPLE_REVIEW_COUNT = 10
EXAMPLE_REVIEW_SEED = 42
example_raw_indices = (
    test_review_df['raw_index']
    .drop_duplicates()
    .sample(n=min(EXAMPLE_REVIEW_COUNT, test_review_df['raw_index'].nunique()), random_state=EXAMPLE_REVIEW_SEED)
)

main_review_predictions_example = main_review_predictions[
    main_review_predictions['raw_index'].isin(example_raw_indices)
].copy()

example_base = (
    main_review_predictions_example
    .drop_duplicates('raw_index')
    [['raw_index', 'rating', 'review_text', 'actual_label_name']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'rating': '별점',
        'review_text': '리뷰',
        'actual_label_name': '실제',
    })
)

proposed_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'proposed']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '제안예측',
        'event_prob': '제안확률',
        'is_correct': '제안정답',
    })
)

selected_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'selected']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '선택예측',
        'event_prob': '선택확률',
        'is_correct': '선택정답',
    })
)

main_review_predictions_simple = (
    example_base
    .merge(proposed_example, on='리뷰번호', how='left')
    .merge(selected_example, on='리뷰번호', how='left')
    .sort_values('리뷰번호')
    .round({
        '제안확률': 4,
        '선택확률': 4,
    })
)

print('리뷰 10개 이벤트 판별 예시: 제안 모델 + 선택 모델')
display(main_review_predictions_simple)


리뷰 10개 이벤트 판별 예시: 제안 모델 + 선택 모델


,리뷰번호,별점,리뷰,실제,제안예측,제안확률,제안정답,선택예측,선택확률,선택정답
0,1202,5.0,급하게 먹느라 사진을 못찍었습니다 ㅠㅠ \n쌀국수 좋아해서 자주 먹는데 여기 쌀국수는 국물부터가 너무 찐하고 너무 맛있어요 양도 많고 자주 애용하겠습니다 ! 진짜 너무 맛있게 잘먹었어요 !,일반 리뷰(0),일반 예측(0),0.0430,True,일반 예측(0),0.4254,True
1,1277,4.0,배달이 좀 늦게왔지만\n맛있게 잘먹었어용,일반 리뷰(0),일반 예측(0),0.0150,True,일반 예측(0),0.2572,True
2,1699,5.0,일주일에 한두번은 시켜먹는곳이예요~!!\n고구마치즈돈까스랑 치즈폭탄 둘이 제일 베스트 메뉴입니당,일반 리뷰(0),이벤트 예측(1),0.8944,False,이벤트 예측(1),0.5289,False
3,1973,5.0,배부르고 맛있게 잘 먹었습니다~갈비만두랑 같이먹으니 더 맛있어요! 꼭 같이 시켜드세요,이벤트 리뷰(1),이벤트 예측(1),0.7788,True,이벤트 예측(1),0.5661,True
4,2816,5.0,짜장면 먹고 싶어서 배달시켰는데 진짜 간짜장식이네요 다른데는 야채가 잘게 썰어 있는데 큼지막하게 되어 있어서 넘 좋았어요 짬뽕국물도 주셔서 먹어봤는데 국물이 얼큰하네요 담엔 짬뽕 먹어보려구요 \n찐짜장면 맛집...,일반 리뷰(0),이벤트 예측(1),0.8695,False,이벤트 예측(1),0.5083,False
5,2911,5.0,젤라또가 원래 가성비 좋은 디저트는 아닌지라 가격대비 양은 적어보이는데 맛은 확실히 좋네요 피스타치오 추천,일반 리뷰(0),이벤트 예측(1),0.5359,False,일반 예측(0),0.4354,True
6,5107,5.0,밀봉도 잘 되어있고 다들 맛있다고 좋아하셨어용🫶🏻,일반 리뷰(0),일반 예측(0),0.1893,True,이벤트 예측(1),0.5481,False
7,5619,5.0,맛있게 먹다가 사진을 못찍었어요ㅠㅠ 건더기 정말 푸짐하고 특히 전 순대가 정말 맛있었어요!! 금데 엄청 매워가지구 다음에는 매운거 말고 일반국밥 시키려구요!! 제육도 완전 밥도둑이어써용!!,일반 리뷰(0),일반 예측(0),0.4511,True,일반 예측(0),0.4252,True
8,5945,5.0,맛있어서 피자 파스타 콜라까지 순삭했어요. 도우가 기름지지않고 끝이 바삭해서 맛있었고 배달 정말 빨랐어요. 잘먹었습니다 :),이벤트 리뷰(1),일반 예측(0),0.4302,False,이벤트 예측(1),0.5265,True
9,7765,5.0,배달도 빠르고 너무 맛있어용\n포장도 완전 잘 해주셨어요,일반 리뷰(0),일반 예측(0),0.1271,True,이벤트 예측(1),0.5361,False


## 6. 후보 1/2 별점 정제 함수 정의

가게별 원래 평균 별점을 기준으로 두 가지 정제 후보를 적용한다.

- 후보 1: 이벤트 리뷰로 예측된 리뷰를 제외하고 평균 계산
- 후보 2: 이벤트 확률이 높을수록 낮은 가중치를 적용해 평균 계산


In [58]:
def original_store_rating(frame):
    return (
        frame.groupby('store_id')
        .agg(
            store_name=('store_name', 'first'),
            review_count=('rating', 'size'),
            original_mean_rating=('rating', 'mean'),
        )
        .reset_index()
    )


def candidate_1_exclude_predicted_event(frame):
    original = original_store_rating(frame)
    kept = frame[frame['event_pred'] == 0]

    refined = (
        kept.groupby('store_id')
        .agg(
            kept_review_count=('rating', 'size'),
            refined_rating=('rating', 'mean'),
        )
        .reset_index()
    )

    result = original.merge(refined, on='store_id', how='left')
    result['fallback_used'] = result['refined_rating'].isna()
    result['kept_review_count'] = result['kept_review_count'].fillna(0).astype(int)
    result['refined_rating'] = result['refined_rating'].fillna(result['original_mean_rating'])
    return result


def candidate_2_probability_weighted(frame):
    work = frame.copy()
    work['weight'] = work['candidate_2_weight']
    work['weighted_rating'] = work['candidate_2_weighted_rating']

    original = original_store_rating(work)
    weighted = (
        work.groupby('store_id')
        .agg(
            weight_sum=('weight', 'sum'),
            weighted_rating_sum=('weighted_rating', 'sum'),
        )
        .reset_index()
    )
    result = original.merge(weighted, on='store_id', how='left')
    result['fallback_used'] = result['weight_sum'].fillna(0) <= 0
    result['refined_rating'] = np.where(
        result['fallback_used'],
        result['original_mean_rating'],
        result['weighted_rating_sum'] / result['weight_sum'],
    )
    return result


def summarize_refinement(store_result, frame, model_info, candidate_key, candidate_label):
    merged = store_result.copy()
    merged['rating_delta'] = merged['refined_rating'] - merged['original_mean_rating']
    return {
        'model_key': model_info['model_key'],
        'model_label': model_info['model_label'],
        'model_role': model_info['model_role'],
        'candidate': candidate_key,
        'candidate_label': candidate_label,
        'threshold': float(frame['threshold'].iloc[0]),
        'store_count': int(merged['store_id'].nunique()),
        'review_count': int(len(frame)),
        'event_pred_count': int(frame['event_pred'].sum()),
        'event_pred_rate': float(frame['event_pred'].mean()),
        'original_mean_rating': float(frame['rating'].mean()),
        'refined_mean_rating': float(merged['refined_rating'].mean()),
        'mean_rating_delta': float(merged['rating_delta'].mean()),
        'mean_abs_rating_delta': float(merged['rating_delta'].abs().mean()),
        'max_abs_rating_delta': float(merged['rating_delta'].abs().max()),
        'fallback_store_count': int(merged['fallback_used'].sum()),
    }


## 7. 전체 모델에 후보 1/2 일괄 적용

4개 모델 각각에 후보 1과 후보 2를 모두 적용한다.


In [51]:
store_results = []
summary_rows = []

candidate_defs = [
    ('candidate_1_exclude_predicted_event', '후보 1: 이벤트 예측 리뷰 제외', candidate_1_exclude_predicted_event),
    ('candidate_2_probability_weighted', '후보 2: 이벤트 확률 기반 가중 평균', candidate_2_probability_weighted),
]

for spec in model_specs:
    frame = prediction_detail[prediction_detail['model_key'] == spec['model_key']].copy()
    for candidate_key, candidate_label, func in candidate_defs:
        store_result = func(frame)
        store_result['model_key'] = spec['model_key']
        store_result['model_label'] = spec['model_label']
        store_result['model_role'] = spec['model_role']
        store_result['candidate'] = candidate_key
        store_result['candidate_label'] = candidate_label
        store_result['rating_delta'] = store_result['refined_rating'] - store_result['original_mean_rating']
        store_results.append(store_result)
        summary_rows.append(summarize_refinement(store_result, frame, spec, candidate_key, candidate_label))

all_store_results = pd.concat(store_results, ignore_index=True)
refinement_summary = pd.DataFrame(summary_rows)

assert len(refinement_summary) == 8, '4개 모델 x 2개 후보 = 8개 결과여야 한다.'

refinement_summary_display = refinement_summary.round({
    'threshold': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('전체 모델 x 정제 후보 결과 수:', len(refinement_summary_display))


전체 모델 x 정제 후보 결과 수: 8


## 8. 리뷰별 예측 결과와 가게별 정제 별점 연결

특정 리뷰를 봤을 때 이 리뷰가 이벤트로 판단됐는지뿐 아니라, 그 판단 이후 해당 가게의 평균 별점이 후보 1/2에서 어떻게 바뀌었는지도 같이 확인한다.

여기서는 이후 계산에 필요한 연결 테이블만 만든다. 리뷰별 화면 출력은 5번 예시 표로 제한한다.


In [52]:
candidate_1_store = (
    all_store_results[all_store_results['candidate'] == 'candidate_1_exclude_predicted_event']
    [[
        'model_key',
        'store_id',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
    ]]
    .rename(columns={
        'original_mean_rating': 'original_store_rating',
        'refined_rating': 'candidate_1_store_refined_rating',
        'rating_delta': 'candidate_1_store_rating_delta',
    })
)

candidate_2_store = (
    all_store_results[all_store_results['candidate'] == 'candidate_2_probability_weighted']
    [[
        'model_key',
        'store_id',
        'refined_rating',
        'rating_delta',
    ]]
    .rename(columns={
        'refined_rating': 'candidate_2_store_refined_rating',
        'rating_delta': 'candidate_2_store_rating_delta',
    })
)

review_refinement_detail = (
    prediction_detail
    .merge(candidate_1_store, on=['model_key', 'store_id'], how='left')
    .merge(candidate_2_store, on=['model_key', 'store_id'], how='left')
)

assert len(review_refinement_detail) == len(prediction_detail)

review_refinement_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'is_correct',
    'error_type',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_1_store_rating_delta',
    'candidate_2_store_refined_rating',
    'candidate_2_store_rating_delta',
]

simple_refinement_cols = [
    'model_label',
    'rating',
    'review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_2_store_refined_rating',
]

simple_refinement_column_names = {
    'model_label': '모델',
    'rating': '리뷰별점',
    'review_text': '리뷰',
    'actual_label_name': '실제',
    'event_pred_name': '예측',
    'event_prob': '이벤트확률',
    'candidate_1_action': '후보1',
    'candidate_2_weight': '후보2가중치',
    'original_store_rating': '기존가게별점',
    'candidate_1_store_refined_rating': '후보1정제별점',
    'candidate_2_store_refined_rating': '후보2정제별점',
}

main_review_refinement_detail = (
    review_refinement_detail[review_refinement_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_refinement_full = main_review_refinement_detail[review_refinement_cols].round({
    'event_prob': 4,
    'candidate_2_weight': 4,
    'original_store_rating': 4,
    'candidate_1_store_refined_rating': 4,
    'candidate_1_store_rating_delta': 4,
    'candidate_2_store_refined_rating': 4,
    'candidate_2_store_rating_delta': 4,
})

main_review_refinement_simple = (
    main_review_refinement_detail[simple_refinement_cols]
    .rename(columns=simple_refinement_column_names)
    .round({
        '이벤트확률': 4,
        '후보2가중치': 4,
        '기존가게별점': 4,
        '후보1정제별점': 4,
        '후보2정제별점': 4,
    })
)

print('리뷰별 예측과 가게 정제 별점 연결 테이블 생성 완료:', main_review_refinement_simple.shape)


리뷰별 예측과 가게 정제 별점 연결 테이블 생성 완료: (2652, 11)


## 9. 특정 가게의 전체 평점 정제 검증

별점 변화가 크게 나타난 가게 하나를 가져와 후보 1/2 계산이 어떻게 적용되는지 확인한다.

- 후보 1: 이벤트로 예측된 리뷰를 제외한 뒤 평균 계산
- 후보 2: `1 - event_prob`를 가중치로 둔 가중 평균 계산

`STORE_EXAMPLE_ID`에 특정 `store_id`를 넣으면 자동 선택이 아니라 해당 가게를 직접 확인할 수 있다. 여기서는 해당 가게의 전체 리뷰 목록은 출력하지 않고, 평점 정제 검산표만 출력한다.


In [53]:
STORE_EXAMPLE_ID = None
STORE_EXAMPLE_MODEL_ROLES = ['proposed', 'selected']

store_change_pool = (
    all_store_results[all_store_results['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)]
    .assign(abs_rating_delta=lambda df: df['rating_delta'].abs())
    .sort_values('abs_rating_delta', ascending=False)
    .reset_index(drop=True)
)

if STORE_EXAMPLE_ID is None:
    selected_store_id = store_change_pool.iloc[0]['store_id']
else:
    selected_store_id = STORE_EXAMPLE_ID

store_example_detail = review_refinement_detail[
    review_refinement_detail['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)
    & review_refinement_detail['store_id'].eq(selected_store_id)
].copy()

if store_example_detail.empty:
    raise ValueError(f'선택한 가게가 test 데이터에 없습니다: {selected_store_id}')

manual_rows = []
for (model_key, model_label), group in store_example_detail.groupby(['model_key', 'model_label']):
    original_mean = group['rating'].mean()
    kept = group[group['event_pred'] == 0]
    candidate_1_manual = original_mean if kept.empty else kept['rating'].mean()
    weight_sum = group['candidate_2_weight'].sum()
    candidate_2_manual = (
        original_mean
        if weight_sum <= 0
        else (group['rating'] * group['candidate_2_weight']).sum() / weight_sum
    )
    first = group.iloc[0]
    manual_rows.append({
        '모델': model_label,
        '리뷰수': len(group),
        '이벤트예측수': int(group['event_pred'].sum()),
        '후보1포함수': int((group['event_pred'] == 0).sum()),
        '기존가게별점': original_mean,
        '후보1직접계산': candidate_1_manual,
        '후보1저장값': first['candidate_1_store_refined_rating'],
        '후보1변화량': candidate_1_manual - original_mean,
        '후보2직접계산': candidate_2_manual,
        '후보2저장값': first['candidate_2_store_refined_rating'],
        '후보2변화량': candidate_2_manual - original_mean,
        '후보2평균가중치': group['candidate_2_weight'].mean(),
    })

store_rating_refinement_example = pd.DataFrame(manual_rows).round({
    '기존가게별점': 4,
    '후보1직접계산': 4,
    '후보1저장값': 4,
    '후보1변화량': 4,
    '후보2직접계산': 4,
    '후보2저장값': 4,
    '후보2변화량': 4,
    '후보2평균가중치': 4,
})

store_name_for_print = store_example_detail['store_name'].iloc[0]
print('선택 가게:', store_name_for_print)
print('store_id:', selected_store_id)
print('후보 1/2 직접 계산값과 저장된 정제값 비교')
display(store_rating_refinement_example)


선택 가게: https://s.baemin.com/1l000fQNHPpDv
store_id: https://s.baemin.com/1l000fQNHPpDv
후보 1/2 직접 계산값과 저장된 정제값 비교


,모델,리뷰수,이벤트예측수,후보1포함수,기존가게별점,후보1직접계산,후보1저장값,후보1변화량,후보2직접계산,후보2저장값,후보2변화량,후보2평균가중치
0,선택 모델 (Metadata only + MLP),11,9,2,4.6364,3.0000,3.0000,-1.6364,4.3147,4.3147,-0.3217,0.5100
1,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),11,5,6,4.6364,4.3333,4.3333,-0.3030,4.3766,4.3766,-0.2598,0.5822


## 10. 메인 비교표: 제안 모델 vs 선택 모델

가게별 별점 정제 결과의 최종 해석은 제안 모델과 선택 모델 중심으로 진행한다.


In [54]:
main_roles = ['proposed', 'selected']
main_comparison = (
    refinement_summary[refinement_summary['model_role'].isin(main_roles)]
    .sort_values(['model_role', 'candidate'])
    .reset_index(drop=True)
)

main_comparison_display = main_comparison[[
    'model_label',
    'candidate_label',
    'threshold',
    'review_count',
    'event_pred_count',
    'event_pred_rate',
    'store_count',
    'original_mean_rating',
    'refined_mean_rating',
    'mean_rating_delta',
    'mean_abs_rating_delta',
    'max_abs_rating_delta',
    'fallback_store_count',
]].round({
    'threshold': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('메인 비교: 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) vs 선택 모델 (Metadata only + MLP)')
display(main_comparison_display)


메인 비교: 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) vs 선택 모델 (Metadata only + MLP)


,model_label,candidate_label,threshold,review_count,event_pred_count,event_pred_rate,store_count,original_mean_rating,refined_mean_rating,mean_rating_delta,mean_abs_rating_delta,max_abs_rating_delta,fallback_store_count
0,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),후보 1: 이벤트 예측 리뷰 제외,0.5074,1326,580,0.4374,118,4.8989,4.8505,-0.0256,0.0404,0.4500,0
1,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),후보 2: 이벤트 확률 기반 가중 평균,0.5074,1326,580,0.4374,118,4.8989,4.8458,-0.0303,0.0381,0.4699,0
2,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,0.5004,1326,843,0.6357,118,4.8989,4.7656,-0.1105,0.1114,1.6364,2
3,선택 모델 (Metadata only + MLP),후보 2: 이벤트 확률 기반 가중 평균,0.5004,1326,843,0.6357,118,4.8989,4.8316,-0.0445,0.0445,0.5729,0


## 11. 부록 전체 비교표

전체 모델 4개에 후보 2개를 모두 적용한 결과다.

이 표는 부록/참고용으로 사용하고, 최종 해석은 메인 비교표를 중심으로 한다.


In [55]:
appendix_comparison_display = refinement_summary[[
    'model_label',
    'model_role',
    'candidate_label',
    'threshold',
    'review_count',
    'event_pred_count',
    'event_pred_rate',
    'store_count',
    'original_mean_rating',
    'refined_mean_rating',
    'mean_rating_delta',
    'mean_abs_rating_delta',
    'max_abs_rating_delta',
    'fallback_store_count',
]].sort_values(['model_role', 'model_label', 'candidate_label']).reset_index(drop=True).round({
    'threshold': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('부록: 전체 모델 4개 x 정제 후보 2개')
display(appendix_comparison_display)


부록: 전체 모델 4개 x 정제 후보 2개


,model_label,model_role,candidate_label,threshold,review_count,event_pred_count,event_pred_rate,store_count,original_mean_rating,refined_mean_rating,mean_rating_delta,mean_abs_rating_delta,max_abs_rating_delta,fallback_store_count
0,KcBERT PCA only + MLP,ablation,후보 1: 이벤트 예측 리뷰 제외,0.5038,1326,525,0.3959,118,4.8989,4.8628,-0.0133,0.0358,0.8333,0
1,KcBERT PCA only + MLP,ablation,후보 2: 이벤트 확률 기반 가중 평균,0.5038,1326,525,0.3959,118,4.8989,4.8691,-0.0070,0.0234,0.2968,0
2,TF-IDF + Random Forest,baseline,후보 1: 이벤트 예측 리뷰 제외,0.5021,1326,200,0.1508,118,4.8989,4.8669,-0.0091,0.0264,0.5250,0
3,TF-IDF + Random Forest,baseline,후보 2: 이벤트 확률 기반 가중 평균,0.5021,1326,200,0.1508,118,4.8989,4.8729,-0.0031,0.0151,0.2423,0
4,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),proposed,후보 1: 이벤트 예측 리뷰 제외,0.5074,1326,580,0.4374,118,4.8989,4.8505,-0.0256,0.0404,0.4500,0
5,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),proposed,후보 2: 이벤트 확률 기반 가중 평균,0.5074,1326,580,0.4374,118,4.8989,4.8458,-0.0303,0.0381,0.4699,0
6,선택 모델 (Metadata only + MLP),selected,후보 1: 이벤트 예측 리뷰 제외,0.5004,1326,843,0.6357,118,4.8989,4.7656,-0.1105,0.1114,1.6364,2
7,선택 모델 (Metadata only + MLP),selected,후보 2: 이벤트 확률 기반 가중 평균,0.5004,1326,843,0.6357,118,4.8989,4.8316,-0.0445,0.0445,0.5729,0


## 12. 오답 리뷰 샘플

FP/FN 오답 리뷰를 확인한다.

- FP: 일반 리뷰인데 이벤트 리뷰로 예측
- FN: 이벤트 리뷰인데 일반 리뷰로 예측


In [56]:
error_samples = (
    review_refinement_detail[
        review_refinement_detail['model_role'].isin(['proposed', 'selected'])
        & review_refinement_detail['error_type'].isin(['FP', 'FN'])
    ]
    .sort_values(['model_role', 'error_type', 'event_prob'], ascending=[True, True, False])
    .groupby(['model_label', 'error_type'], group_keys=False)
    .head(10)
)

error_sample_cols = [
    'model_label',
    'error_type',
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_2_store_refined_rating',
]

print('제안 모델/선택 모델 FP/FN 오답 샘플')
display(error_samples[error_sample_cols].round({
    'event_prob': 4,
    'candidate_2_weight': 4,
    'original_store_rating': 4,
    'candidate_1_store_refined_rating': 4,
    'candidate_2_store_refined_rating': 4,
}))


제안 모델/선택 모델 FP/FN 오답 샘플


,model_label,error_type,raw_index,store_name,rating,review_text,cleaned_review_text,actual_label_name,event_pred_name,event_prob,candidate_1_action,candidate_2_weight,original_store_rating,candidate_1_store_refined_rating,candidate_2_store_refined_rating
3396,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,7671,콩불 건대점,5.0,맛있게 잘먹었습니다~,맛있게 잘먹었습니다~,이벤트 리뷰(1),일반 예측(0),0.5059,포함,0.4941,5.0000,5.0000,5.0000
3739,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,6785,라파리나,5.0,리뷰가 좋길래 시켜봤는데 진짜 맛집인 거 같아요!! 다 너무 맛있어요🥰,리뷰가 좋길래 시켜봤는데 진짜 맛집인 거 같아요!! 다 너무 맛있어요,이벤트 리뷰(1),일반 예측(0),0.5024,포함,0.4976,4.8333,4.7500,4.7020
3072,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,453,https://s.baemin.com/Q6000jyZx2LMo,5.0,맛있어요!,맛있어요!,이벤트 리뷰(1),일반 예측(0),0.5010,포함,0.4990,5.0000,5.0000,5.0000
2946,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,3286,난곡반점,5.0,맛있어용!!\n남은건 내일 밥말아서 또 먹을거에요~~\n짜장밥 감사합니다. 잘먹었습니다 :),맛있어용!! 남은건 내일 밥말아서 또 먹을거에요~~ 짜장밥 감사합니다. 잘먹었습니다 :),이벤트 리뷰(1),일반 예측(0),0.4992,포함,0.5008,5.0000,5.0000,5.0000
2794,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,2607,플러스82 마곡점,5.0,와플은 여기밖에 안 먹어요…\n생크림이 진짜 너무 맛있고 아메리카노 진짜 미쳤어요 생크림이 진짜… 킥..🥰,와플은 여기밖에 안 먹어요 생크림이 진짜 너무 맛있고 아메리카노 진짜 미쳤어요 생크림이 진짜 킥..,이벤트 리뷰(1),일반 예측(0),0.4922,포함,0.5078,5.0000,5.0000,5.0000
3120,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,5168,https://s.baemin.com/Ha000fnAkEtl5,5.0,맛잇긴한데 빵에 비해 속재료는 적은거같아용 ㅠ,맛잇긴한데 빵에 비해 속재료는 적은거같아용 ㅠ,이벤트 리뷰(1),일반 예측(0),0.4903,포함,0.5097,4.7000,4.6667,4.6253
3871,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,3593,드디어 찾은 인생 최고의 중국집,5.0,배달도 빠르고 재료 하나하나가 깔끔해서 맛있었어요. 파김치랑 조합이 정말 좋네요. 잘 먹었습니다.,배달도 빠르고 재료 하나하나가 깔끔해서 맛있었어요. 파김치랑 조합이 정말 좋네요. 잘 먹었습니다.,이벤트 리뷰(1),일반 예측(0),0.4842,포함,0.5158,4.9412,4.9167,4.8974
3420,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,5427,https://s.baemin.com/Kc000fdcXtdfF,5.0,맛있게잘먹었습니다!,맛있게잘먹었습니다!,이벤트 리뷰(1),일반 예측(0),0.4739,포함,0.5261,4.1250,4.0000,3.8906
2906,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,3783,청담과 청담동1% 과일제작소,5.0,정말 너무너무너무 맛있어요!! \n벌써 두번시켜먹었는데 두번다 완전 맛있게 잘 먹었습니다!! 감사합니다!!,정말 너무너무너무 맛있어요!! 벌써 두번시켜먹었는데 두번다 완전 맛있게 잘 먹었습니다!! 감사합니다!!,이벤트 리뷰(1),일반 예측(0),0.4671,포함,0.5329,5.0000,5.0000,5.0000
3386,제안 모델 (Hybrid MLP: KcBERT PCA + Metadata),FN,4218,https://s.baemin.com/rA000fdJDbNmv,5.0,따뜻하고 넘 맛잇어용 이근처 파스타 맛집 못찾아 아쉬웠는데 요기 찾은것같아 다행이에여🫶 맛나게먹엇습니당!,따뜻하고 넘 맛잇어용 이근처 파스타 맛집 못찾아 아쉬웠는데 요기 찾은것같아 다행이에여 맛나게먹엇습니당!,이벤트 리뷰(1),일반 예측(0),0.4571,포함,0.5429,5.0000,5.0000,5.0000


## 13. 별점 변화가 큰 가게 샘플

제안 모델과 선택 모델에서 별점 변화가 큰 가게를 확인한다.


In [57]:
top_changed_samples = (
    all_store_results[all_store_results['model_role'].isin(['proposed', 'selected'])]
    .assign(abs_rating_delta=lambda df: df['rating_delta'].abs())
    .sort_values('abs_rating_delta', ascending=False)
    [[
        'model_label',
        'candidate_label',
        'store_id',
        'store_name',
        'review_count',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
        'fallback_used',
    ]]
    .head(20)
    .round({
        'original_mean_rating': 4,
        'refined_rating': 4,
        'rating_delta': 4,
    })
)

display(top_changed_samples)


,model_label,candidate_label,store_id,store_name,review_count,original_mean_rating,refined_rating,rating_delta,fallback_used
711,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/1l000fQNHPpDv,https://s.baemin.com/1l000fQNHPpDv,11,4.6364,3.0000,-1.6364,False
724,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Ns000gJNQSvNT,https://s.baemin.com/Ns000gJNQSvNT,13,4.3846,3.4000,-0.9846,False
722,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Kc000fdcXtdfF,https://s.baemin.com/Kc000fdcXtdfF,8,4.1250,3.2500,-0.8750,False
760,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,국민낙곱새 강서점,국민낙곱새 강서점,13,4.6154,3.7500,-0.8654,False
709,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/08000fAyEYHfz,https://s.baemin.com/08000fAyEYHfz,8,4.5000,3.6667,-0.8333,False
741,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/iE000iHSmeDVY,https://s.baemin.com/iE000iHSmeDVY,4,4.2500,3.5000,-0.7500,False
862,선택 모델 (Metadata only + MLP),후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/pG000flG524ai,https://s.baemin.com/pG000flG524ai,10,3.7000,3.1271,-0.5729,False
744,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/pG000flG524ai,https://s.baemin.com/pG000flG524ai,10,3.7000,3.1429,-0.5571,False
840,선택 모델 (Metadata only + MLP),후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/Kc000fdcXtdfF,https://s.baemin.com/Kc000fdcXtdfF,8,4.1250,3.5949,-0.5301,False
733,선택 모델 (Metadata only + MLP),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Vt000hXSsKGTk,https://s.baemin.com/Vt000hXSsKGTk,13,4.7692,4.2500,-0.5192,False


## 14. 최종 해석

07번에서는 test 리뷰별 이벤트 예측 결과와 가게별 별점 정제 결과를 함께 확인한다.

해석 기준은 다음과 같다.

- 개별 리뷰의 별점 자체를 바꾸는 것이 아니라, 해당 리뷰가 속한 가게의 평균 별점 계산에서 제외되거나 낮은 가중치로 반영된다.
- 후보 1은 이벤트 리뷰로 예측된 리뷰를 제외하므로, threshold와 이벤트 예측 개수의 영향을 크게 받는다.
- 후보 2는 이벤트 확률을 연속적인 가중치로 사용하므로, 후보 1보다 변화가 완만할 수 있다.
- 특정 가게 평점 정제 검증 셀을 통해 후보 1/2 계산 결과가 저장된 정제 별점과 일치하는지 확인할 수 있다.
- 선택 모델 (Metadata only + MLP)은 이벤트 리뷰를 많이 잡는 모델이므로 별점 정제 폭이 더 클 수 있다.
- 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)은 선택 모델보다 보수적으로 이벤트 리뷰를 잡기 때문에 별점 변화 폭이 작을 수 있다.
- 발표/보고서에서는 전체 모델 결과를 부록으로 두고, 본문에서는 제안 모델과 선택 모델의 리뷰별 예측 및 후보 1/2 별점 정제 결과를 중심으로 설명한다.

07번 결과를 바탕으로 최종 별점 정제 방식은 다음 중 하나로 정리할 수 있다.

1. 이벤트 리뷰를 강하게 제거하고 싶으면 후보 1을 사용한다.
2. 이벤트 가능성을 부드럽게 반영하고 싶으면 후보 2를 사용한다.
3. 현재 모델의 오탐 가능성을 고려하면 후보 2의 확률 기반 가중 평균이 더 안정적인 정제 방식으로 해석될 수 있다.
